# Group 2 Final Project

* Jussan Nascimento (Data Engineer)
* Nilay Sundarkar (Data Engineer)
* Seraphim Eilken (Data Scientist)
* Nathaniel Ekanem (Data Scientist)
* Josh White (BI Analyst)
* Jeremy Rue (Data Architect)

### Business challenge: 

Corporación Favorita is a large grocery retailer in Ecuador. We are helping them build a Lakehouse and forecasting solution that predicts short-term sales per story and product families. We will expose insights through dashboards so planners can optimize inventory, promotions and sales. 

* **Demand forecasting** - We will predict daily sales for each (`store_nbr`, `family`) for the next XX days. 
* **Promotion effectiveness** - We will provide insights on how `onpromotion` and holiday periods impact sales. 
* **Operational nimbleness** - Provide information for right-size orders and build resilience around holidays, and unforseen events like oil price shocks, earthquakes, etc.)

Dataset here: [https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)

Thie project will design a medallion lakehouse (bronze/silver/gold) with daily refreshing, and provide a baseline model fed from the gold table. Metrics will be surfaced with SQL Dashboards. 


### Catalog and Schemas (BRONZE LAYER)

**stores.csv**

| Column    | Type   | Description                          |
| --------- | ------ | ------------------------------------ |
| store_nbr | INT    | Store identifier (unique per store). |
| city      | STRING | City where the store is located.     |
| state     | STRING | Region/province within Ecuador.      |
| type      | STRING | Store type letter   (e.g., A, B, C). |
| cluster   | INT    | Cluster id grouping similar stores.  |


**transactions.csv** (unused)

| Column       | Type | Description                                     |
| ------------ | ---- | ----------------------------------------------- |
| date         | DATE | Transaction date.                               |
| store_nbr    | INT  | Store identifier.                               |
| transactions | INT  | Num of transactions for that store on that date.|


**holidays.csv**

| Column      | Type    | Description                                               |
| ----------- | ------- | --------------------------------------------------------- |
| date        | DATE    | Date associated with the event/holiday.                   |
| type        | STRING  | Event type (Holiday, Additional, Bridge, Transfer, etc.). |
| locale      | STRING  | Scope: National, Regional, Local.                         |
| locale_name | STRING  | Name of the region/city for the holiday.                  |
| description | STRING  | Human-readable event description.                         |
| transferred | BOOLEAN | Whether this holiday was transferred to another date.     |


**oil.csv**

| Column     | Type   | Description                        |
| ---------- | ------ | ---------------------------------- |
| date       | DATE   | Date of the oil price observation. |
| dcoilwtico | DOUBLE | Daily oil price index.             |


**train.csv**
**test.csv** (unused test.csv)

| Column      | Type   | Description                                                        |
| ----------- | ------ | ------------------------------------------------------------------ |
| id          | INT    | Unique row id from Kaggle training set.                            |
| date        | DATE   | Transaction/sales date.                                            |
| store_nbr   | INT    | Store identifier.                                                  |
| family      | STRING | Product family/category (e.g., GROCERY I, AUTOMOTIVE).             |
| sales       | DOUBLE | Unit sales for that store/family/date.                             |
| onpromotion | INT    | Number of items on promotion for this `(store_nbr, family, date)`. |


### Star Schema (Gold Layer)

<img src="https://lh3.google.com/u/0/d/1yeukCCietWDqoIGWc6RyyrbxpRFXgXZ-=w2952-h2080-iv1?auditContext=prefetch" alt="EDR diagram showing start schema for gold layer">

Bronze Layer: Ingest CSVs into Bronze Delta tables

* train_sales_raw (bronze.train_sales_raw)
* stores_raw (bronze.stores_raw)
* holidays_raw (bronze.holidays_raw)
* oil_raw (bronze.oil_raw)
* ~~test_sales_raw~~ not needed, disregard
* ~~transactions_raw~~ not needed, disregard

Build Silver tables

* Create silver.train_sales, silver.stores, silver.holidays, silver.oil
  - cast types `date` to DATE, `store_nbr` to INT, `sales` to DOUBLE, etc.
  - filter out invalid negative sales
* Resolve any nulls, standardize dates

Build Gold tables

* Create dimension tables. Maybe we can call them `dim_date`, `dim_store`, `dim_family`, `dim_holidays`, `dim_oil`
  - The dim_store, dim_family, dim_date all come from silver.train_sales
  - The dim_oil comes from silver.oil
  - The dim_holidays comes from silver.holidays
* Each dimension table should have unique primary key, i.e. `store_key`, `family_key` as a surrogate key.
* For the date dimension
  - One row per calendar day
  - add a surrogate key `date_key`
  - add calendar attributes (day of week, month, year, etc.) SQL has `dayofweek(date)` methods, etc.
* Build `fact_daily_sales` table, one row per (date_key, store_key, family_key)
* Tables must support MERGE so that we can add additional data
* Suggest partition on fact_daily_sales table on time, then z-order by store_key, family_key. 

Streaming requirement

* Use a streaming job with trigger(once=True)
* Read Silver fact-incremental then write to Gold fact

Data Scientists:

* Predict **sales** column (y) from FACT_DAILY_SALES for each store + family + date (only use Gold, clean star schema)
* Use all other features to predict the sales column. 
* Ignore sales_fact_id (key not predictave) and created_at (ETL timestamp, irrelevant)
* Use date_key, store_key, family_key to join to other tables (e.g. `fact_daily_sales JOIN date ON fact.date_key = date.date_key`)

BI person:

* create dashboards at the end. 

#### Questions to ask:

1. Do we need everything in one big cell, or have separate cells? Everything in a big notebook?
2. Should we have one notebook or several notebooks?
3. Do we need to create a real pipeline, or just demonstrate how it would work?
4. Do we need to use the visual pipeline diagram?
5. Project deliverables? 
6. What streaming tables would we use (i.e. silver to gold?)


In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS store_sales_catalog
COMMENT "Catalog for the Final project.";
USE CATALOG store_sales_catalog;

CREATE SCHEMA IF NOT EXISTS raw
COMMENT "Raw layer: contains only Volumes with original source files exactly as downloaded.";
USE SCHEMA raw;

CREATE VOLUME IF NOT EXISTS store_sales_vol
COMMENT "Volume for storing raw Kaggle CSV files.";

CREATE SCHEMA IF NOT EXISTS bronze
COMMENT "Bronze layer: minimally processed Delta tables loaded from raw files.";

CREATE SCHEMA IF NOT EXISTS silver
COMMENT "Silver layer: cleaned and conformed Delta tables with standardized data, business rules, and joins applied.";

CREATE SCHEMA IF NOT EXISTS gold
COMMENT "Gold layer: curated business-ready tables for analytics, reporting, dashboards, and ML outputs.";

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6012624604990300>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE CATALOG IF NOT EXISTS store_sales_catalog\nCOMMENT "Catalog for the Final project.";\nUSE CATALOG store_sales_catalog;\n\nCREATE SCHEMA IF NOT EXISTS raw\nCOMMENT "Raw layer: contains only Volumes with original source files exactly as downloaded.";\nUSE SCHEMA raw;\n\nCREATE VOLUME IF NOT EXISTS store_sales_vol\nCOMMENT "Volume for storing raw Kaggle CSV files.";\n\nCREATE SCHEMA IF NOT EXISTS bronze\nCOMMENT "Bronze layer: minimally processed Delta tables loaded from raw files.";\n\nCREATE SCHEMA IF NOT EXISTS silver\nCOMMENT "Silver layer: cleaned and conformed Delta tables with standardized data, business rules, and joins applied.";\n\nCREATE SCHEMA IF NOT EXISTS gold\nCOMMENT "Gold layer: curated business-ready tables for analytics, repor

In [0]:
# Set up paths
catalog = "store_sales_catalog"
raw_path = f"/Volumes/{catalog}/raw/store_sales_vol"
bronze_schema = f"{catalog}.bronze" 
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

In [0]:
# Create folders to add Kaggle CSV files
dbutils.fs.mkdirs(f"{raw_path}/train")
dbutils.fs.mkdirs(f"{raw_path}/stores")
dbutils.fs.mkdirs(f"{raw_path}/oil")
dbutils.fs.mkdirs(f"{raw_path}/transactions")
dbutils.fs.mkdirs(f"{raw_path}/holidays_events")

# Creat folders for Autoload checkpoints and schemas 
dbutils.fs.mkdirs(f"{raw_path}/_chk_train")
dbutils.fs.mkdirs(f"{raw_path}/_schema_train")

dbutils.fs.mkdirs(f"{raw_path}/_chk_stores")
dbutils.fs.mkdirs(f"{raw_path}/_schema_stores")

dbutils.fs.mkdirs(f"{raw_path}/_chk_transactions")
dbutils.fs.mkdirs(f"{raw_path}/_schema_transactions")

dbutils.fs.mkdirs(f"{raw_path}/_chk_oil")
dbutils.fs.mkdirs(f"{raw_path}/_schema_oil")

dbutils.fs.mkdirs(f"{raw_path}/_chk_holidays_events")
dbutils.fs.mkdirs(f"{raw_path}/_schema_holidays_events")

dbutils.fs.mkdirs(f"{raw_path}/_chk_gold_fact")

True

Bronze layer

In [0]:
%sql
USE CATALOG store_sales_catalog;
USE SCHEMA bronze;

-- SALES
CREATE TABLE IF NOT EXISTS train_bronze (
  id INT,
  date STRING,
  store_nbr INT,
  family STRING,
  sales DOUBLE,
  onpromotion INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (id)
  -- FOREIGN KEY (store_nbr) REFERENCES stores_bronze(store_nbr)
  -- FOREIGN KEY (date) REFERENCES oil_bronze(date)
  -- FOREIGN KEY (date) REFERENCES transactions_bronze(date)
  -- FOREIGN KEY (date) REFERENCES holidays_events_bronze(date)
)
USING DELTA;

-- STORES
CREATE TABLE IF NOT EXISTS stores_bronze (
  store_nbr INT,
  city STRING,
  state STRING,
  type STRING,
  cluster INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (store_nbr)
)
USING DELTA;

-- OIL
CREATE TABLE IF NOT EXISTS oil_bronze (
  date STRING,
  dcoilwtico DOUBLE,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date)
)
USING DELTA;

-- TRANSACTIONS
CREATE TABLE IF NOT EXISTS transactions_bronze (
  date STRING,
  store_nbr INT,
  transactions INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date, store_nbr)
  -- FOREIGN KEY (store_nbr) REFERENCES stores_bronze(store_nbr)
  -- FOREIGN KEY (date) REFERENCES oil_bronze(date)
  -- FOREIGN KEY (date) REFERENCES holidays_events_bronze(date)
)
USING DELTA;

-- HOLIDAYS
CREATE TABLE IF NOT EXISTS holidays_events_bronze (
  date STRING,
  type STRING,
  locale STRING,
  locale_name STRING,
  description STRING,
  transferred STRING,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date, type, locale, locale_name, description)
)
USING DELTA;


Autoloader

In [0]:
from pyspark.sql.functions import *

catalog = "store_sales_catalog"
bronze_schema = "bronze"
raw_path = "/Volumes/store_sales_catalog/raw/store_sales_vol"
chk_path = f"{raw_path}/_chk"
schema_path = f"{raw_path}/_schema"

def autoload_csv(input_path, table_name, checkpoint_name, schema_name):
    (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{schema_path}/{schema_name}")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(input_path)
        .withColumn("_source_file", 
            col("_metadata.file_path")
        )
        .withColumn("_ingest_ts", current_timestamp())
        .writeStream
        .trigger(once=True)
        .option("checkpointLocation", f"{chk_path}/{checkpoint_name}")
        .option("mergeSchema", "true")
        .toTable(table_name)
    )

autoload_csv(
    input_path=f"{raw_path}/train",
    table_name=f"{catalog}.{bronze_schema}.train_bronze",
    checkpoint_name="train",
    schema_name="train"
)

autoload_csv(
    input_path=f"{raw_path}/stores",
    table_name=f"{catalog}.{bronze_schema}.stores_bronze",
    checkpoint_name="stores",
    schema_name="stores"
)

autoload_csv(
    input_path=f"{raw_path}/oil",
    table_name=f"{catalog}.{bronze_schema}.oil_bronze",
    checkpoint_name="oil",
    schema_name="oil"
)

autoload_csv(
    input_path=f"{raw_path}/transactions",
    table_name=f"{catalog}.{bronze_schema}.transactions_bronze",
    checkpoint_name="transactions",
    schema_name="transactions"
)

autoload_csv(
    input_path=f"{raw_path}/holidays_events",
    table_name=f"{catalog}.{bronze_schema}.holidays_events_bronze",
    checkpoint_name="holidays_events",
    schema_name="holidays_events"
)


Validation 

In [0]:
%sql
SELECT COUNT(*), MIN(_ingest_ts), MAX(_ingest_ts)
FROM store_sales_catalog.bronze.train_bronze;



COUNT(*),MIN(_ingest_ts),MAX(_ingest_ts)
3000888,2025-11-28T18:51:05.108Z,2025-11-28T18:51:05.108Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.train_bronze LIMIT 10

id,date,store_nbr,family,sales,onpromotion,_source_file,_ingest_ts,_rescued_data
0,2013-01-01,1,AUTOMOTIVE,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
1,2013-01-01,1,BABY CARE,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
2,2013-01-01,1,BEAUTY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
3,2013-01-01,1,BEVERAGES,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
4,2013-01-01,1,BOOKS,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
5,2013-01-01,1,BREAD/BAKERY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
6,2013-01-01,1,CELEBRATION,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
7,2013-01-01,1,CLEANING,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
8,2013-01-01,1,DAIRY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
9,2013-01-01,1,DELI,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null


In [0]:
%sql

SELECT * FROM store_sales_catalog.bronze.stores_bronze LIMIT 10;


store_nbr,city,state,type,cluster,_source_file,_ingest_ts,_rescued_data
1,Quito,Pichincha,D,13,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
2,Quito,Pichincha,D,13,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
3,Quito,Pichincha,D,8,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
4,Quito,Pichincha,D,9,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
5,Santo Domingo,Santo Domingo de los Tsachilas,D,4,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
6,Quito,Pichincha,D,13,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
7,Quito,Pichincha,D,8,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
8,Quito,Pichincha,D,8,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
9,Quito,Pichincha,B,6,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null
10,Quito,Pichincha,C,15,/Volumes/store_sales_catalog/raw/store_sales_vol/stores/stores.csv,2025-11-28T18:51:06.307Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.oil_bronze LIMIT 10;


date,dcoilwtico,_source_file,_ingest_ts,_rescued_data
2013-01-01,null,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-02,93.14,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-03,92.97,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-04,93.12,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-07,93.2,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-08,93.21,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-09,93.08,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-10,93.81,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-11,93.6,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-14,94.27,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.transactions_bronze LIMIT 10;


date,store_nbr,transactions,_source_file,_ingest_ts,_rescued_data
2013-01-01,25,770,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,1,2111,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,2,2358,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,3,3487,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,4,1922,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,5,1903,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,6,2143,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,7,1874,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,8,3250,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,9,2940,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.holidays_events_bronze LIMIT 10;

date,type,locale,locale_name,description,transferred,_source_file,_ingest_ts,_rescued_data
2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null


Exploratory analysis before creating Silver Layer

In [0]:
# Null values
def null_report(df):
    return df.select([
          sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])


df_train = spark.table("store_sales_catalog.bronze.train_bronze")
null_report(df_train).display()

id,date,store_nbr,family,sales,onpromotion,_source_file,_ingest_ts,_rescued_data
0,0,0,0,0,0,0,0,3000888


In [0]:

# Summary statistcs

df_train.summary().display()



summary,id,date,store_nbr,family,sales,onpromotion,_source_file,_rescued_data
count,3000888,3000888,3000888,3000888,3000888,3000888,3000888,0
mean,1500443.5,null,27.5,null,357.7757491126198,2.6027702466736513,null,null
stddev,866281.8916415141,null,15.585786717870711,null,1101.9977213379998,12.218882346424381,null,null
min,0,2013-01-01,1,AUTOMOTIVE,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,null
25%,750057,null,14,null,0.0,0,null,null
50%,1500406,null,28,null,11.0,0,null,null
75%,2250636,null,41,null,195.957,0,null,null
max,3000887,2017-08-15,54,SEAFOOD,124717.0,741,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,null
